In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from pandas.tseries.holiday import USFederalHolidayCalendar

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

df = pd.read_csv("data/daily_requests.csv", parse_dates=["day"])
s = df.set_index("day")["requests"]

In [5]:
def make_features(s):
    F = pd.DataFrame(index=s.index)
    F["lag1"] = s.shift(1)                       # вчера
    F["lag7"] = s.shift(7)                       # неделю назад
    F["lag14"] = s.shift(14)                     # две недели назад
    F["mean7"] = s.shift(1).rolling(7).mean()    # среднее за прошлые 7 дней

    holidays = USFederalHolidayCalendar().holidays(s.index.min(), s.index.max())
    F["is_holiday"] = s.index.isin(holidays).astype(int)

    day = pd.get_dummies(s.index.dayofweek, prefix="day", drop_first=True).astype(int)
    day.index = s.index
    return F.join(day)

X = make_features(s)
data = X.join(s.rename("y")).dropna()
print(data.shape)
data.head()

(1812, 12)


,lag1,lag7,lag14,mean7,is_holiday,day_1,day_2,day_3,day_4,day_5,day_6,y
day,,,,,,,,,,,,
2021-01-15,7592.0,7628.0,5482.0,7436.428571,0,0,0,0,1,0,0,7097
2021-01-16,7097.0,6799.0,5933.0,7360.571429,0,0,0,0,0,1,0,6365
2021-01-17,6365.0,6591.0,5564.0,7298.571429,0,0,0,0,0,0,1,5977
2021-01-18,5977.0,8207.0,7885.0,7210.857143,1,0,0,0,0,0,0,6635
2021-01-19,6635.0,7803.0,8243.0,6986.285714,0,1,0,0,0,0,0,8967


In [3]:
train = data[:"2024-12-31"]
test = data["2025-01-01":]

features = [c for c in data.columns if c != "y"]
model = LinearRegression().fit(train[features], train["y"])
pred = model.predict(test[features])

print(f"Линейная регрессия: MAE = {mae(test['y'], pred):.0f}, MAPE = {mape(test['y'], pred):.1f}%")
print(f"Baseline 'как вчера': MAE = {mae(test['y'], test['lag1']):.0f}, "
      f"MAPE = {mape(test['y'], test['lag1']):.1f}%")

Линейная регрессия: MAE = 651, MAPE = 6.3%
Baseline 'как вчера': MAE = 814, MAPE = 8.1%
